# 🧪 ActiveMQ Client Demo - Whitelisted vs Non-Whitelisted Access

This notebook demonstrates how Azure Firewall controls access to ActiveMQ running in Container Apps.

## What This Proves

| Test | Source IP | Firewall Rule | Expected Result |
|------|-----------|---------------|------------------|
| ✅ Test 1 | Whitelisted IP | DNAT → ActiveMQ | **Connection SUCCESS** |
| ✅ Test 1b | Whitelisted IP | STOMP send/recv | **Message round-trip** |
| ❌ Test 2 | Non-whitelisted IP | No matching rule | **Connection REFUSED/TIMEOUT** |
| 📊 Test 3 | Side-by-side | Compare results | **Allow vs Deny** |

## Architecture

```mermaid
graph LR
    Client["🧑‍💻 Python Client<br/>(This Notebook)"] -->|":61616 STOMP"| FW["🔥 Azure Firewall<br/>(Public IP)"]
    FW -->|"DNAT"| AMQ["📦 ActiveMQ<br/>(Internal ACA)"]

    FW -->|"Checks src IP<br/>against whitelist"| Decision{{"IP in<br/>whitelist?"}}

    Decision -->|"✅ Yes"| Allow["✅ DNAT to<br/>ActiveMQ"]
    Decision -->|"❌ No"| Drop["❌ DROP<br/>(timeout)"]

    style Client fill:#e3f2fd,stroke:#1565c0,stroke-width:2px
    style FW fill:#ff9800,color:#fff,stroke:#e65100,stroke-width:2px
    style AMQ fill:#4caf50,color:#fff,stroke:#2e7d32,stroke-width:2px
    style Decision fill:#fff3e0,stroke:#ff9800,stroke-width:2px
    style Allow fill:#4caf50,color:#fff,stroke:#2e7d32
    style Drop fill:#ef5350,color:#fff,stroke:#c62828
```

### Azure Firewall SNAT Behavior (Egress)

When ActiveMQ **sends responses back**, the Firewall performs SNAT:
- Response packets exit via the Firewall's Public IP
- The client sees the Firewall's IP as the source, NOT the container's private IP

### Monitoring

An **Azure Monitor Dashboard** and **Log Analytics Workbook** provide:
- Denied traffic timeline & top blocked source IPs
- DNAT allowed connections & rule hit counts
- Firewall health, throughput, and data processed metrics

> Access via: **Azure Portal → Dashboard** or **Azure Portal → Monitor → Workbooks**

## Prerequisites

In [1]:
# Verify required packages are available
# If stomp is missing, run in terminal: python -m pip install stomp.py
try:
    import stomp
    print(f"✅ stomp.py {stomp.__version__} installed")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "stomp.py"])
    import stomp
    print(f"✅ stomp.py {stomp.__version__} installed (just now)")

print("✅ Dependencies ready")

✅ stomp.py 8.2.0 installed
✅ Dependencies ready


In [ ]:
# =============================================================================
# Configuration
# =============================================================================
import socket
import time
import urllib.request
import ssl
import json
import subprocess

# 🔑 Firewall Public IP (auto-detected from deployment outputs)
FIREWALL_PUBLIC_IP = None

try:
    result = subprocess.run(
        'az deployment group show -g rg-activemq-firewall-mvp -n main --query properties.outputs.firewallPublicIp.value -o tsv',
        shell=True, capture_output=True, text=True
    )
    if result.returncode == 0 and result.stdout.strip():
        FIREWALL_PUBLIC_IP = result.stdout.strip()
        print(f"🎯 Firewall Public IP: {FIREWALL_PUBLIC_IP}")
except Exception:
    pass

if not FIREWALL_PUBLIC_IP:
    FIREWALL_PUBLIC_IP = "20.230.16.44"  # Fallback - update if re-deployed
    print(f"⚠️  Using fallback Firewall IP: {FIREWALL_PUBLIC_IP}")

TARGET_HOST = FIREWALL_PUBLIC_IP

# ActiveMQ ports (STOMP on 61616 since ACA only exposes single TCP port)
STOMP_PORT = 61616
WEB_CONSOLE_PORT = 8161

# ActiveMQ credentials (default for rmohr/activemq image)
AMQ_USER = "admin"
AMQ_PASS = "admin"

# Detect our public IP
try:
    my_ip = urllib.request.urlopen('https://api.ipify.org').read().decode('utf8')
    print(f"📍 Your Public IP: {my_ip}")
except Exception:
    my_ip = "unknown"
    print("⚠️  Could not determine public IP")

print(f"\n🎯 Target: {TARGET_HOST}:{STOMP_PORT}")
print(f"   Traffic flow: {my_ip} → Firewall ({FIREWALL_PUBLIC_IP}) → ActiveMQ")

📍 Your Public IP: 98.118.154.232
   This is what Azure Firewall sees as your source IP

🎯 Target: 20.110.61.7:61616
   Traffic flow: 98.118.154.232 → Firewall (20.110.61.7) → ActiveMQ


## ✅ Test 1: TCP Connection from Whitelisted Source

If your IP is in the Firewall's DNAT whitelist, this connection should **succeed**.

In [8]:
# =============================================================================
# Test 1: TCP Socket Connection Test (Whitelisted Source)
# =============================================================================
# This tests raw TCP connectivity through the Firewall to ActiveMQ
# Port 61616 now serves STOMP protocol (not OpenWire)
# If whitelisted: Connection succeeds
# If not whitelisted: Connection times out (Firewall drops packets)
# =============================================================================

def test_tcp_connection(host: str, port: int, timeout: int = 10) -> dict:
    """Test TCP connectivity to a host:port and return results."""
    result = {
        "host": host,
        "port": port,
        "success": False,
        "banner": None,
        "error": None,
        "latency_ms": None,
    }
    
    start = time.time()
    try:
        sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        sock.settimeout(timeout)
        
        print(f"   🔌 Connecting to {host}:{port} (timeout: {timeout}s)...")
        sock.connect((host, port))
        
        latency = (time.time() - start) * 1000
        result["latency_ms"] = round(latency, 2)
        result["success"] = True
        
        # STOMP is a request-response protocol; server waits for client CONNECT frame.
        # Send a minimal STOMP CONNECT to verify the service responds.
        try:
            sock.settimeout(5)
            stomp_connect = b'CONNECT\naccept-version:1.2\nhost:localhost\n\n\x00'
            sock.sendall(stomp_connect)
            banner = sock.recv(1024)
            if b'CONNECTED' in banner:
                result["banner"] = "ActiveMQ STOMP Protocol Detected"
            else:
                result["banner"] = f"Received {len(banner)} bytes (unexpected response)"
        except socket.timeout:
            result["banner"] = "Connected but no STOMP response (timeout)"
        
        sock.close()
        
    except socket.timeout:
        result["error"] = "Connection TIMED OUT (Firewall likely blocking - packet dropped)"
    except ConnectionRefusedError:
        result["error"] = "Connection REFUSED (port closed or service not running)"
    except ConnectionResetError:
        result["error"] = "Connection RESET (Firewall actively rejecting)"
    except Exception as e:
        result["error"] = f"Error: {type(e).__name__}: {str(e)}"
    
    return result


# Run Test 1: STOMP Port
print("=" * 70)
print("  TEST 1: Whitelisted Client → Azure Firewall → ActiveMQ (TCP 61616)")
print("=" * 70)
print(f"\n📍 Source IP: {my_ip} (should be in Firewall whitelist)")
print(f"🎯 Target:   {TARGET_HOST}:{STOMP_PORT}\n")

result = test_tcp_connection(TARGET_HOST, STOMP_PORT, timeout=15)

if result["success"]:
    print(f"\n   ✅ CONNECTION SUCCESSFUL!")
    print(f"   ⏱️  Latency: {result['latency_ms']}ms")
    print(f"   📨 Banner: {result['banner']}")
    print(f"\n   → This proves your IP ({my_ip}) IS whitelisted in Azure Firewall")
    print(f"   → Firewall DNAT forwarded traffic to ActiveMQ inside the VNet")
else:
    print(f"\n   ❌ CONNECTION FAILED: {result['error']}")
    print(f"\n   → If TIMEOUT: Your IP ({my_ip}) is likely NOT in the whitelist")
    print(f"   → If REFUSED: ActiveMQ may not be running or port not mapped")

  TEST 1: Whitelisted Client → Azure Firewall → ActiveMQ (TCP 61616)

📍 Source IP: 98.118.154.232 (should be in Firewall whitelist)
🎯 Target:   20.110.61.7:61616

   🔌 Connecting to 20.110.61.7:61616 (timeout: 15s)...

   ✅ CONNECTION SUCCESSFUL!
   ⏱️  Latency: 766.84ms
   📨 Banner: Connected but no banner (OK for some protocols)

   → This proves your IP (98.118.154.232) IS whitelisted in Azure Firewall
   → Firewall DNAT forwarded traffic to ActiveMQ inside the VNet


## ✅ Test 1b: STOMP Protocol Message Test (Whitelisted)

Send an actual message to ActiveMQ using the STOMP protocol on port 61616.
ActiveMQ is configured to serve STOMP on port 61616 (since ACA TCP ingress only exposes a single port).

In [9]:
# =============================================================================
# Test 1b: Send/Receive Message via STOMP Protocol
# =============================================================================
# STOMP is a simple text-based protocol supported by ActiveMQ.
# ActiveMQ is configured to serve STOMP on port 61616 (replacing OpenWire)
# because ACA TCP ingress only supports a single exposed port.
# =============================================================================

import stomp
import threading

class ActiveMQListener(stomp.ConnectionListener):
    """Listener that captures received messages."""
    def __init__(self):
        self.messages = []
        self.errors = []
        self.connected = False
        self.event = threading.Event()
    
    def on_connected(self, frame):
        self.connected = True
        print(f"   ✅ STOMP Connected to ActiveMQ")
        print(f"      Server: {frame.headers.get('server', 'N/A')}")
        print(f"      Version: {frame.headers.get('version', 'N/A')}")
    
    def on_message(self, frame):
        self.messages.append({
            "body": frame.body,
            "destination": frame.headers.get('destination', 'N/A'),
            "message_id": frame.headers.get('message-id', 'N/A'),
            "timestamp": frame.headers.get('timestamp', 'N/A'),
        })
        self.event.set()
    
    def on_error(self, frame):
        self.errors.append(frame.body)
        print(f"   ❌ STOMP Error: {frame.body}")


print("=" * 70)
print("  TEST 1b: STOMP Message Send/Receive (Whitelisted Client)")
print("=" * 70)

QUEUE_NAME = "/queue/firewall-test"
TEST_MESSAGE = f"Hello from whitelisted client! IP: {my_ip}, Time: {time.strftime('%Y-%m-%d %H:%M:%S')}"

try:
    # Connect via STOMP on port 61616
    conn = stomp.Connection([(TARGET_HOST, STOMP_PORT)])
    listener = ActiveMQListener()
    conn.set_listener('test', listener)
    
    print(f"\n   🔌 Connecting via STOMP to {TARGET_HOST}:{STOMP_PORT}...")
    conn.connect(AMQ_USER, AMQ_PASS, wait=True)
    
    # Subscribe to queue
    conn.subscribe(destination=QUEUE_NAME, id='1', ack='auto')
    print(f"   📥 Subscribed to queue: {QUEUE_NAME}")
    
    # Send message
    conn.send(destination=QUEUE_NAME, body=TEST_MESSAGE, content_type='text/plain')
    print(f"   📤 Sent message: {TEST_MESSAGE}")
    
    # Wait for message
    listener.event.wait(timeout=10)
    
    if listener.messages:
        msg = listener.messages[0]
        print(f"\n   ✅ MESSAGE RECEIVED SUCCESSFULLY!")
        print(f"      Body:        {msg['body']}")
        print(f"      Queue:       {msg['destination']}")
        print(f"      Message ID:  {msg['message_id']}")
        print(f"\n   🔒 This proves:")
        print(f"      1. Azure Firewall allowed our IP ({my_ip})")
        print(f"      2. DNAT forwarded TCP traffic to ActiveMQ container")
        print(f"      3. ActiveMQ served STOMP protocol on port {STOMP_PORT}")
        print(f"      4. Response came back through Firewall SNAT")
    else:
        print(f"   ⚠️  Message sent but not received back (timeout)")
    
    conn.disconnect()
    print(f"   🔌 Disconnected")

except stomp.exception.ConnectFailedException as e:
    print(f"\n   ❌ STOMP connection failed: {e}")
    print(f"      Verify ActiveMQ is running with STOMP transport on port {STOMP_PORT}")
except Exception as e:
    print(f"\n   ❌ Error: {type(e).__name__}: {e}")

  TEST 1b: STOMP Message Send/Receive (Whitelisted Client)

   🔌 Connecting via STOMP to 20.110.61.7:61613...

   ❌ STOMP connection failed: 
      Firewall may be blocking STOMP port 61613
      Ensure DNAT rule exists for port 61613


## ❌ Test 2: Simulated Non-Whitelisted Source

This demonstrates what happens when a **non-whitelisted** IP tries to connect.

### How Azure Firewall Blocks Non-Whitelisted Traffic

```mermaid
sequenceDiagram
    participant Client as 🚫 Non-Whitelisted Client<br/>(IP: 192.168.99.99)
    participant FW as 🔥 Azure Firewall<br/>(DNAT Rules)

    Client->>FW: TCP SYN → FW_PIP:61616
    FW->>FW: Check DNAT rules
    Note over FW: src IP 192.168.99.99<br/>NOT in whitelist!
    FW--xClient: ❌ PACKET DROPPED (no response)
    Note over Client: ⏳ ... timeout ...
    Client->>Client: ✗ Connection times out
```

> **Note**: We can't actually change our source IP, so we demonstrate this by:
> 1. Showing the TCP timeout behavior from a non-whitelisted perspective
> 2. Querying Azure Firewall logs for denied traffic

In [ ]:
# =============================================================================
# Test 2: Simulate Non-Whitelisted Connection Attempt
# =============================================================================
# Since we can't spoof our IP, we demonstrate the firewall deny behavior.
# In production, a non-whitelisted client would see exactly this timeout.
# =============================================================================

print("=" * 70)
print("  TEST 2: Non-Whitelisted Client → Azure Firewall → ❌ BLOCKED")
print("=" * 70)

# Method 1: Connect to a port NOT covered by DNAT rules
# This simulates what a non-whitelisted IP would experience
NON_DNAT_PORT = 9999  # Port with no DNAT rule

print(f"\n📍 Testing connection to a non-DNAT'd port (simulates non-whitelisted)")
print(f"🎯 Target: {FIREWALL_PUBLIC_IP}:{NON_DNAT_PORT}")
print(f"   Expected: Timeout (firewall drops packet silently)\n")

start_time = time.time()
result = test_tcp_connection(FIREWALL_PUBLIC_IP, NON_DNAT_PORT, timeout=10)
elapsed = time.time() - start_time

if not result["success"]:
    print(f"\n   ❌ CONNECTION BLOCKED (as expected!)")
    print(f"   ⏱️  Waited: {elapsed:.1f}s before timeout")
    print(f"   📋 Error: {result['error']}")
    print(f"\n   🔒 This is exactly what a non-whitelisted IP would experience:")
    print(f"      • Firewall receives the TCP SYN packet")
    print(f"      • No matching DNAT rule for source IP or dest port")
    print(f"      • Packet is SILENTLY DROPPED (no RST, no ICMP)")
    print(f"      • Client times out waiting for SYN-ACK")
    print(f"      • Attacker learns nothing about internal infrastructure")
else:
    print(f"\n   ⚠️  Unexpected: Connection succeeded on port {NON_DNAT_PORT}")

In [ ]:
# =============================================================================
# Test 2b: Query Firewall Logs for Denied Connections
# =============================================================================
# Shows actual deny entries from Azure Firewall's network rule logs
# This proves the firewall is actively blocking non-whitelisted traffic
#
# NOTE: Diagnostic settings use AzureDiagnostics mode, so logs are in the
#       AzureDiagnostics table filtered by Category, NOT in resource-specific
#       tables like AZFWNetworkRule.
# =============================================================================

import subprocess

RESOURCE_GROUP = "rg-activemq-firewall-mvp"  # Same as deployment
LOG_ANALYTICS_WORKSPACE = None  # Will be auto-detected

print("=" * 70)
print("  TEST 2b: Azure Firewall Deny Logs (Proof of Blocking)")
print("=" * 70)

# Find Log Analytics workspace
try:
    result = subprocess.run(
        f'az monitor log-analytics workspace list --resource-group {RESOURCE_GROUP} -o json',
        shell=True, capture_output=True, text=True
    )
    workspaces = json.loads(result.stdout)
    if workspaces:
        LOG_ANALYTICS_WORKSPACE = workspaces[0]['customerId']
        print(f"\n   📊 Log Analytics Workspace: {workspaces[0]['name']}")
except Exception:
    pass

if LOG_ANALYTICS_WORKSPACE:
    # KQL query for denied connections in the last 1 hour
    # Uses AzureDiagnostics table (not resource-specific AZFWNetworkRule)
    kql_query = """
    AzureDiagnostics
    | where Category == "AzureFirewallNetworkRule"
    | where TimeGenerated > ago(1h)
    | where msg_s has "Deny"
    | project TimeGenerated, msg_s
    | order by TimeGenerated desc
    | take 20
    """
    
    try:
        result = subprocess.run(
            f'az monitor log-analytics query '
            f'--workspace {LOG_ANALYTICS_WORKSPACE} '
            f'--analytics-query "{kql_query}" -o json',
            shell=True, capture_output=True, text=True
        )
        
        logs = json.loads(result.stdout) if result.stdout.strip() else []
        
        if logs:
            print(f"\n   🔥 Denied Connections (Last 1 Hour):")
            print(f"   {'Time':<22} {'Details'}")
            print(f"   {'-'*22} {'-'*60}")
            for log in logs[:10]:
                msg = log.get('msg_s', 'N/A')
                # Truncate long messages
                if len(msg) > 60:
                    msg = msg[:60] + "..."
                print(f"   {log.get('TimeGenerated', 'N/A'):<22} ❌ {msg}")
            print(f"\n   → These entries prove the Firewall is blocking non-whitelisted IPs")
        else:
            print(f"\n   ℹ️  No deny logs found in the last hour.")
            print(f"      Logs may take 5-10 minutes to appear in Log Analytics.")
    except Exception as e:
        print(f"\n   ⚠️  Could not query logs: {e}")
else:
    print(f"\n   ⚠️  Log Analytics workspace not found. Deploy infrastructure first.")
    print(f"      To manually query, use Azure Portal > Log Analytics > Logs")
    print(f"      KQL Query:")
    print(f"      AzureDiagnostics | where Category == 'AzureFirewallNetworkRule' | where msg_s has 'Deny' | take 20")

## 📊 Test 3: Compare Whitelisted vs Non-Whitelisted (Side-by-Side)

In [ ]:
# =============================================================================
# Test 3: Side-by-Side Comparison
# =============================================================================

print("=" * 70)
print("  COMPARISON: Whitelisted vs Non-Whitelisted Access")
print("=" * 70)

tests = [
    {
        "name": "✅ Whitelisted (STOMP :61616)",
        "host": TARGET_HOST,
        "port": STOMP_PORT,
        "desc": "DNAT rule matches → forwarded to ActiveMQ"
    },
    {
        "name": "✅ Whitelisted (Web Console :8161)",
        "host": TARGET_HOST,
        "port": WEB_CONSOLE_PORT,
        "desc": "DNAT rule matches → forwarded to ActiveMQ Web UI"
    },
    {
        "name": "❌ Non-Whitelisted (Random Port :9999)",
        "host": FIREWALL_PUBLIC_IP,
        "port": 9999,
        "desc": "No DNAT rule → DROPPED by Firewall"
    },
    {
        "name": "❌ Non-Whitelisted (SSH :22)",
        "host": FIREWALL_PUBLIC_IP,
        "port": 22,
        "desc": "No DNAT rule → DROPPED by Firewall"
    },
]

print(f"\n{'Test':<45} {'Result':<12} {'Latency':<12} {'Details'}")
print(f"{'-'*45} {'-'*12} {'-'*12} {'-'*30}")

for test in tests:
    result = test_tcp_connection(test["host"], test["port"], timeout=8)
    status = "✅ ALLOWED" if result["success"] else "❌ BLOCKED"
    latency = f"{result['latency_ms']}ms" if result["latency_ms"] else "timeout"
    detail = result["banner"] if result["success"] else result["error"][:30]
    print(f"{test['name']:<45} {status:<12} {latency:<12} {detail}")

print(f"\n{'='*70}")
print(f"  CONCLUSION:")
print(f"  • Azure Firewall DNAT rules selectively forward whitelisted traffic")
print(f"  • Non-matching traffic is silently dropped (no information leaked)")
print(f"  • All egress from ActiveMQ exits via Firewall's Public IP (SNAT)")
print(f"  • This provides a stable, predictable, auditable egress identity")
print(f"{'='*70}")

## 🏗️ Architecture Summary for Customer

### Security Controls in Place

| Layer | Control | Purpose |
|-------|---------|--------|
| **Perimeter** | Azure Firewall DNAT | Only whitelisted IPs can reach ActiveMQ |
| **Network** | VNet + Internal ACA | ActiveMQ not directly exposed to internet |
| **Egress** | UDR + Firewall SNAT | All outbound traffic exits via known IP |
| **Monitoring** | Dashboard + Workbook | Real-time visibility into allowed/denied traffic |
| **Audit** | Firewall Logs + Log Analytics | Full audit trail of all connection attempts |
| **Default Deny** | Firewall implicit deny | Everything not explicitly allowed is blocked |

### Azure Monitor Dashboard & Workbook

| Resource | Purpose |
|----------|---------|
| **Portal Dashboard** | Quick overview: Firewall health, throughput, denied/allowed connections, traffic by port |
| **Log Analytics Workbook** | Deep drill-down: Interactive time range, denied sources, DNAT timeline, raw logs |

> Access via: **Azure Portal → Dashboards** or **Azure Portal → Monitor → Workbooks**

### Egress IP Guarantee

| Configuration | Behavior |
|--------------|----------|
| **1 Firewall Public IP** | All outbound = that IP (deterministic) |
| **Multiple Public IPs** | Outbound uses any of them (Azure selects) |
| **IP Prefix** | Use Public IP Prefix for contiguous range |

### Why This Architecture?

1. **Known Egress IP** - Partners/APIs can whitelist your Firewall's Public IP
2. **IP Whitelisting** - Only approved clients can reach ActiveMQ
3. **No Direct Exposure** - ActiveMQ has no public IP; only accessible via Firewall
4. **Full Audit** - Every connection attempt is logged (allowed + denied)
5. **Visual Monitoring** - Dashboard and Workbook for quick troubleshooting
6. **Compliance** - Meets enterprise security requirements for message brokers

### 💰 Estimated Monthly Cost (Per Environment)

| Resource | SKU/Tier | Monthly Cost |
|----------|----------|-------------|
| Azure Firewall | **Basic** | ~$288 |
| Public IPs (×2) | Standard | ~$7 |
| Container Apps | Consumption | ~$78 |
| Log Analytics | Per-GB | ~$5–15 |
| VNet / Route Table / NSG | — | Free |
| Dashboard / Workbook | — | Free |
| **Total per environment** | | **~$380–390/mo** |
| **4 environments** | | **~$1,520–1,560/mo** |

> See the deploy notebook for optimization tips (shared firewall, reserved instances).